# V38 — Restore age from the previous demographic source

Run before the adapted HMM estimator. Read `processed/demographics/household_demographics_2008_2017.tsv` directly, using the exact age construction from `05_build_hmm_estimation_panel_skimmed_no_extra_three_year_filter.ipynb`. No previous trip-level estimation Parquet is required.

**Definition:** for each head, use birth-derived age when 18–110 inclusive; otherwise use the age-code mapping 1→22, 2→27, 3→32, 4→37, 5→42, 6→47, 7→52, 8→59.5, 9→70. Other codes have no band fallback. The under-25 and 65+ values (22 and 70) are inherited assumptions. Average the two available head ages, or use the sole available one. No additional calendar-year shift is applied.

Required source columns: `hhid`, `panel_year`, `male_head_age_from_birth`, `female_head_age_from_birth`, `Male_Head_Age`, `Female_Head_Age`. The demographic source is configurable; select the source corresponding to the annual panel.

Write a new `hmm_annual_covariates_income_imputed_with_age.parquet`. Preserve all input household-years and original covariates. Duplicate demographic keys, unmatched records or unresolved ages stop execution; no families are silently dropped. Requires duckdb, pandas and pyarrow.


In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import time

DATA_ROOT = Path(r'C:/Users/rwen13/Documents/healthy_purchase_data')
MASTER_ROOT = DATA_ROOT / 'intermediate/v38_milk_beverage_combined/06_hmm_master'
ANNUAL_INPUT = MASTER_ROOT / 'hmm_annual_covariates_income_imputed.parquet'
# Same demographic source and age rule as the previous estimation-panel builder.
AGE_SOURCE = DATA_ROOT / 'processed/demographics/household_demographics_2008_2017.tsv'
# If using the new V38 demographic extraction instead, set AGE_SOURCE explicitly to:
# DATA_ROOT / 'intermediate/v38_milk_beverage_combined/02_demographics/household_demographics_2008_2017.tsv'
# These demographic years are already aligned; do not shift them again.
ANNUAL_OUTPUT = MASTER_ROOT / 'hmm_annual_covariates_income_imputed_with_age.parquet'
AUDIT_DIR = MASTER_ROOT / 'validation/age_recovery'
TEMP_DIR = DATA_ROOT / 'temp/v38_milk_beverage_combined/age_recovery'
YEARS = range(2008, 2018)
for folder in [AUDIT_DIR, TEMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
for path in [ANNUAL_INPUT, AGE_SOURCE]:
    if not path.is_file():
        raise FileNotFoundError(f'Required file missing: {path}')
assert ANNUAL_OUTPUT not in [ANNUAL_INPUT, AGE_SOURCE]

def qp(path):
    return "'" + Path(path).as_posix().replace("'", "''") + "'"

con = duckdb.connect(':memory:')
con.execute("SET threads=1")
con.execute("SET memory_limit='4GB'")
con.execute('SET preserve_insertion_order=false')
con.execute(f'SET temp_directory={qp(TEMP_DIR)}')
con.execute(f'CREATE VIEW annual_original AS SELECT * FROM read_parquet({qp(ANNUAL_INPUT)})')
con.execute(f"""
    CREATE TEMP TABLE age_source AS
    WITH raw AS (
        SELECT CAST(hhid AS VARCHAR) AS hhid,
               TRY_CAST(panel_year AS INTEGER) AS panel_year,
               TRY_CAST(male_head_age_from_birth AS DOUBLE) AS male_birth_age,
               TRY_CAST(female_head_age_from_birth AS DOUBLE) AS female_birth_age,
               TRY_CAST(Male_Head_Age AS INTEGER) AS male_age_code,
               TRY_CAST(Female_Head_Age AS INTEGER) AS female_age_code,

        FROM read_csv({qp(AGE_SOURCE)}, delim='\t', header=true,
                      all_varchar=true)
    ), coded AS (
        SELECT *,
            CASE male_age_code
                WHEN 1 THEN 22.0 WHEN 2 THEN 27.0 WHEN 3 THEN 32.0
                WHEN 4 THEN 37.0 WHEN 5 THEN 42.0 WHEN 6 THEN 47.0
                WHEN 7 THEN 52.0 WHEN 8 THEN 59.5 WHEN 9 THEN 70.0
            END AS male_band_age,
            CASE female_age_code
                WHEN 1 THEN 22.0 WHEN 2 THEN 27.0 WHEN 3 THEN 32.0
                WHEN 4 THEN 37.0 WHEN 5 THEN 42.0 WHEN 6 THEN 47.0
                WHEN 7 THEN 52.0 WHEN 8 THEN 59.5 WHEN 9 THEN 70.0
            END AS female_band_age
        FROM raw
    ), selected AS (
        SELECT *,
               CASE WHEN male_birth_age BETWEEN 18 AND 110 THEN male_birth_age
                    ELSE male_band_age END AS male_age,
               CASE WHEN female_birth_age BETWEEN 18 AND 110 THEN female_birth_age
                    ELSE female_band_age END AS female_age,
               (male_birth_age IS NOT NULL AND NOT (male_birth_age BETWEEN 18 AND 110))
                   AS male_birth_age_invalid,
               (female_birth_age IS NOT NULL AND NOT (female_birth_age BETWEEN 18 AND 110))
                   AS female_birth_age_invalid
        FROM coded
    )
    SELECT hhid, panel_year,
           CASE WHEN male_age IS NOT NULL AND female_age IS NOT NULL
                    THEN (male_age + female_age) / 2.0
                ELSE COALESCE(male_age, female_age) END AS age,
           male_birth_age_invalid, female_birth_age_invalid,
           CASE WHEN male_birth_age_invalid AND male_band_age IS NOT NULL THEN 1 ELSE 0 END
               + CASE WHEN female_birth_age_invalid AND female_band_age IS NOT NULL THEN 1 ELSE 0 END
               AS age_head_fallback_count
    FROM selected
""")

a_cols = {r[0] for r in con.execute('DESCRIBE annual_original').fetchall()}
s_cols = {r[0] for r in con.execute('DESCRIBE age_source').fetchall()}
if not {'hhid', 'panel_year'} <= a_cols or not {'hhid', 'panel_year', 'age'} <= s_cols:
    raise ValueError('Annual input requires hhid/panel_year; age source requires hhid/panel_year/age.')
if 'age' in a_cols:
    raise ValueError('Annual input already contains age. Review it before replacing the age definition.')
con.execute('''CREATE TEMP TABLE target_keys AS
    SELECT CAST(hhid AS VARCHAR) hhid, TRY_CAST(panel_year AS INTEGER) panel_year
    FROM annual_original''')
bad = con.execute('''SELECT COUNT(*) FROM target_keys WHERE hhid IS NULL OR TRIM(hhid)=''
    OR panel_year IS NULL OR panel_year NOT BETWEEN 2008 AND 2017''').fetchone()[0]
dup = con.execute('''SELECT COUNT(*) FROM (
    SELECT hhid,panel_year FROM target_keys GROUP BY ALL HAVING COUNT(*)>1)''').fetchone()[0]
if bad or dup:
    raise ValueError(f'Annual input has invalid keys={bad}, duplicate household-years={dup}')

# Demographics must be one row per household-year; stop on duplicate keys.
if con.execute('SELECT 1 FROM age_source GROUP BY hhid,panel_year HAVING COUNT(*)>1 LIMIT 1').fetchone():
    raise ValueError('Duplicate demographic household-year keys; resolve before matching.')
# The smaller demographic source is parsed once, then matched one target year at a time.
parts = []
for year in YEARS:
    started = time.perf_counter()
    part = TEMP_DIR / f'age_{year}.parquet'
    con.execute(f'''COPY (
        SELECT CAST(s.hhid AS VARCHAR) hhid, {year}::INTEGER panel_year,
               MIN(TRY_CAST(s.age AS DOUBLE)) age,
               COUNT(DISTINCT TRY_CAST(s.age AS DOUBLE)) distinct_age_values,
               COUNT(*) FILTER (WHERE TRY_CAST(s.age AS DOUBLE) IS NULL
                   OR NOT isfinite(TRY_CAST(s.age AS DOUBLE))
                   OR TRY_CAST(s.age AS DOUBLE)<18 OR TRY_CAST(s.age AS DOUBLE)>110) invalid_age_rows,
               COUNT(*) source_demographic_rows
        FROM age_source s SEMI JOIN target_keys k
          ON CAST(s.hhid AS VARCHAR)=k.hhid AND k.panel_year={year}
        WHERE s.panel_year={year}
        GROUP BY CAST(s.hhid AS VARCHAR)
    ) TO {qp(part)} (FORMAT PARQUET, COMPRESSION ZSTD)''')
    parts.append(part)
    print(f'Age extraction {year}: {time.perf_counter()-started:.1f}s', flush=True)
rel='read_parquet(['+','.join(qp(p) for p in parts)+'], hive_partitioning=false)'
con.execute(f'CREATE VIEW recovered_age AS SELECT * FROM {rel}')
con.execute('''CREATE TEMP TABLE age_coverage AS
    SELECT k.hhid,k.panel_year,a.age,a.distinct_age_values,a.invalid_age_rows,a.source_demographic_rows,
      CASE WHEN a.hhid IS NULL THEN 'missing_household_year'
           WHEN a.invalid_age_rows>0 THEN 'invalid_source_age'
           WHEN a.distinct_age_values<>1 THEN 'conflicting_source_age'
           ELSE 'matched' END AS age_status
    FROM target_keys k LEFT JOIN recovered_age a USING(hhid,panel_year)''')
summary=con.execute('''SELECT panel_year,age_status,COUNT(*) household_years
    FROM age_coverage GROUP BY 1,2 ORDER BY 1,2''').df()
summary.to_csv(AUDIT_DIR/'age_matching_summary.csv',index=False)
print(summary.to_string(index=False))
problems=con.execute("SELECT COUNT(*) FROM age_coverage WHERE age_status<>'matched'").fetchone()[0]
if problems:
    con.execute(f'''COPY (SELECT * FROM age_coverage WHERE age_status<>'matched')
        TO {qp(AUDIT_DIR/'age_unresolved_household_years.csv')} (HEADER, DELIMITER ',')''')
    con.close()
    raise ValueError(f'{problems:,} household-years lack one valid source age. See {AUDIT_DIR}. '
                     'No completed annual output was written; no households were dropped. '
                     'Supply a source with the SAME age definition and complete matching coverage.')

pending=ANNUAL_OUTPUT.with_suffix('.pending.parquet')
con.execute(f'''COPY (
    SELECT a.*, r.age FROM annual_original a LEFT JOIN recovered_age r
      ON CAST(a.hhid AS VARCHAR)=r.hhid AND a.panel_year=r.panel_year
    ORDER BY a.hhid,a.panel_year
) TO {qp(pending)} (FORMAT PARQUET, COMPRESSION ZSTD)''')
input_n=con.execute('SELECT COUNT(*) FROM annual_original').fetchone()[0]
output_n=con.execute(f'SELECT COUNT(*) FROM read_parquet({qp(pending)})').fetchone()[0]
assert input_n==output_n, 'Annual row count changed'
con.close()
pending.replace(ANNUAL_OUTPUT)
for part in parts: part.unlink()
print(f'Saved {input_n:,} household-years; age rebuilt using the original demographic rule: {ANNUAL_OUTPUT}')


Age extraction 2008: 0.0s
Age extraction 2009: 0.0s
Age extraction 2010: 0.0s
Age extraction 2011: 0.0s
Age extraction 2012: 0.0s
Age extraction 2013: 0.0s
Age extraction 2014: 0.0s
Age extraction 2015: 0.0s
Age extraction 2016: 0.0s
Age extraction 2017: 0.0s
 panel_year age_status  household_years
       2008    matched            41135
       2009    matched            45553
       2010    matched            47612
       2011    matched            44869
       2012    matched            46053
       2013    matched            46932
       2014    matched            47829
       2015    matched            48622
       2016    matched            43503
       2017    matched            39222
Saved 451,330 household-years; age rebuilt using the original demographic rule: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\06_hmm_master\hmm_annual_covariates_income_imputed_with_age.parquet
